# 未见扰动：幅度、方向和误差
使用末轮 checkpoint 的冻结验证集。该组 Jurkat 验证条件为训练未见扰动；图里的 All / Seen / Unseen 指**输出基因的表达训练暴露**，是另一条轴。无效 Pearson 保留原因，不用 0 填补。

In [ ]:
import os

mode = os.environ.get("GRADPERT_UNSEEN_MODE", "summary")
if mode not in {"summary", "audit", "archive"}:
    raise ValueError(f"Unsupported notebook mode {mode!r}")

In [ ]:
if mode == "summary":
    from analysis_charts import (
        comparison_errors,
        condition_errors,
        effect_diagnostics,
        expression_gap,
        load,
        note,
        pearson_charts,
    )

    data = load("condition-aggregates.json")
    validation = load("MAIN_EXPRESSION_VAL_ACCEPTANCE.json")
    rows = data["rows"]
    pearson_charts(rows, validation)
    expression_gap(rows)
    condition_errors(rows, validation)

## 对照基线与效应强弱
效应以同一批冻结配对对照的均值为零点。训练均值基线来自冻结训练扰动均值，每个训练条件等权。正误差增益代表模型优于对照；用真值 RMS 等频分层只用于定位验证问题。低于等值线表示幅度偏小；幅度接近、余弦仍低则优先检查方向。带目标基因的方向图单独解释，不代表下游效应。参考 [Systema](https://www.nature.com/articles/s41587-025-02777-8.pdf) 的平凡基线设计。

In [ ]:
if mode == "summary":
    if data["response_diagnostics"] is not None:
        effect_diagnostics(rows)
        comparison_errors(rows)
    else:
        note("历史回执未采集响应均值。幅度、方向和基线比较不可用。")

## 冻结基因审计

在服务器用 `GRADPERT_UNSEEN_MODE=audit` 执行原小模型和加深双卡最大 batch 的末轮 checkpoint。基因表达、预测值和逐细胞数据留在服务器。每组沿用已验收的验证条件、对照、真值细胞和表达基因划分。使用 `archive` 在本机导入通过校验的图和排名表。

设预测和真值均减去同一批 300 个配对对照的均值，得到响应 p 和 t。每个基因×扰动的均方误差可拆成幅度项 `(|p|-|t|)^2` 和反号项 `4|pt|·I(pt<0)`。用 `max(|p|-|t|, 0)^2` 单独衡量过度放大，并按基因、扰动和组合排名。另把每个基因的有符号误差分成跨扰动的共同偏移和剩余变化，定位持续偏高或偏低的基因。该拆法在单个基因上定义，前面的方向余弦仍用于整条表达向量。排名是这份冻结验证集的描述性结果。

In [ ]:
if mode in {"audit", "archive"}:
    import hashlib
    import json
    from pathlib import Path

    def sha256(path):
        with path.open("rb") as stream:
            return hashlib.file_digest(stream, "sha256").hexdigest()

    def write_json(path, value):
        temporary = path.with_suffix(".tmp")
        temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False))
        temporary.replace(path)

    def require(condition, message):
        if not condition:
            raise ValueError(message)

    def read_json(path):
        return json.loads(path.read_text())

    job = Path(os.environ["GRADPERT_UNSEEN_JOB"]).resolve()
    arms = ("baseline", "deep-dual")
    labels = {"baseline": "Original small", "deep-dual": "Deep dual max batch"}

In [ ]:
if mode == "audit":
    import gc
    import subprocess
    from unittest.mock import patch

    import numpy as np

    from gradpert.execution.v2_checkpoint_eval import evaluate_worker
    from gradpert.training.v2 import evaluation
    from scripts.v2.replay_small_softmax_frozen import validate_replay
    from scripts.v2.run_group import lock
    from scripts.v2.run_small_softmax import admit_selected_idle

    previous = job.parent / "jurkat-frozen-replay-20261111-a1"
    previous_queue = previous / "QUEUE.json"
    queue_sha = "d2f5c7086cfb8979302a6619d1562492a9549ccb2fb5199789be9df65aeac5c2"
    require(sha256(previous_queue) == queue_sha, "Historical queue hash changed")
    queue = read_json(previous_queue)
    prior_terminal = read_json(previous / "STATE.json")
    require(
        prior_terminal["status"] == "completed" and prior_terminal["queue_sha256"] == queue_sha,
        "Historical replay is not accepted",
    )
    source = Path(evaluation.__file__).resolve().parents[4]
    commit = subprocess.check_output(
        ["git", "-C", str(source), "rev-parse", "HEAD"], text=True
    ).strip()
    require(
        not subprocess.check_output(["git", "-C", str(source), "status", "--porcelain"]),
        "Notebook and evaluator source must be clean",
    )
    require(os.environ["GRADPERT_UNSEEN_COMMIT"] == commit, "Audit source mismatch")
    notebook = source / "notebooks/small_softmax/02_unseen_perturbations.ipynb"
    require(sha256(notebook) == os.environ["GRADPERT_UNSEEN_NOTEBOOK_SHA"], "Notebook hash changed")
    require(
        os.environ.get("CUDA_VISIBLE_DEVICES") == str(queue["gpu_index"])
        and os.environ.get("PYTORCH_ALLOC_CONF") == "expandable_segments:True"
        and os.environ.get("CUBLAS_WORKSPACE_CONFIG") == ":4096:8",
        "GPU or allocator does not match frozen replay",
    )
    gpu_lock = job.parents[1] / "runtime" / f"v2-gpu-{queue['gpu_index']}.lock"

    def admit():
        # The shared lock protects other GraD-Pert jobs. Query the selected GPU's UUID
        # and check any compute PID belongs to this notebook kernel.
        from scripts.v2.shared_gpu import snapshot

        admit_selected_idle(snapshot(os.getpid()), queue["gpu_index"], queue["gpu_uuid"])

    def capture_arm(arm):
        item = next(entry for entry in queue["arms"] if entry["id"] == arm)
        accepted_file = previous / arm / "ACCEPTED.json"
        require(
            sha256(accepted_file) == prior_terminal["accepted"][arm], "Replay acceptance changed"
        )
        accepted = read_json(accepted_file)
        previous_result = previous / arm / "frozen/COMPLETE.json"
        require(sha256(previous_result) == accepted["evaluation_sha256"], "Replay output changed")
        original = read_json(previous_result)
        plan = dict(original["plan"])
        require(
            plan["training_identity"]["run_id"] == item["run_id"]
            and plan["checkpoint_sha256"] == item["checkpoint_sha256"]
            and sha256(Path(plan["checkpoint"])) == item["checkpoint_sha256"]
            and plan["split"] == "val"
            and plan["checkpoint_role"] == "last"
            and plan["checkpoint_epoch"] == 2
            and plan["gpu"] == [str(queue["gpu_index"])]
            and plan["response_diagnostics"]
            and plan["cpu_training_state"]
            and plan["cuda_memory_fraction"] == 0.9,
            "Frozen checkpoint or plan changed",
        )
        config = Path(plan["config"])
        require(sha256(config) == plan["config_sha256"], "Frozen config changed")
        arm_dir = job / arm
        arm_dir.mkdir(exist_ok=False)
        plan["output_root"] = str(arm_dir)
        original_evaluate = evaluation.evaluate
        original_summary = evaluation.summarize_condition
        captured = []
        axes = {}

        def record_axes(model, index, data, reference, **kwargs):
            axes["genes"] = tuple(data.expression_gene_ids)
            axes["conditions"] = tuple(row.condition_id for row in data.control_manifest.draws)
            return original_evaluate(model, index, data, reference, **kwargs)

        def record_condition(**kwargs):
            ix = np.array(kwargs["groups"]["unseen_expression"], dtype=np.int64)
            require(len(ix) > 0 and kwargs["position"] == len(captured), "Frozen row order changed")
            if not captured:
                axes["unseen"] = ix
            else:
                require(np.array_equal(axes["unseen"], ix), "Unseen expression axis changed")
            control = kwargs["paired_control"][ix]
            target = np.isin(ix, kwargs["target_indices"])
            captured.append(
                (
                    kwargs["prediction"][ix] - control,
                    kwargs["truth"][ix] - control,
                    control.copy(),
                    target,
                )
            )
            if len(captured) % 32 == 0:
                admit()
            return original_summary(**kwargs)

        admit()
        with (
            patch.object(evaluation, "evaluate", record_axes),
            patch.object(evaluation, "summarize_condition", record_condition),
        ):
            worker = evaluate_worker(plan, 0)
        observed = {"result": worker["result"], "plan": plan, "zero_pkl": True}
        _, parity = validate_replay(arm, original, observed, arm_dir)
        require(
            len(captured) == len(axes["conditions"]) == 445
            and list(axes["conditions"]) == worker["frozen_condition_order"],
            "Frozen numerical replay differs",
        )
        pred, truth, controls, direct = map(np.stack, zip(*captured, strict=True))
        require(all(np.isfinite(x).all() for x in (pred, truth, controls)), "Nonfinite response")
        require(sha256(Path(plan["checkpoint"])) == item["checkpoint_sha256"], "Checkpoint changed")
        return {
            "pred": pred,
            "truth": truth,
            "controls": controls,
            "direct": direct,
            "genes": [axes["genes"][i] for i in axes["unseen"]],
            "conditions": list(axes["conditions"]),
            "provenance": {
                "run_id": item["run_id"],
                "experiment_id": item["experiment_id"],
                "checkpoint_sha256": item["checkpoint_sha256"],
                "config_sha256": plan["config_sha256"],
                "replay_sha256": accepted["evaluation_sha256"],
                "parity": parity,
                "worker": worker["runtime_measurement"],
            },
        }

    with lock(gpu_lock):
        audit = {}
        for arm in arms:
            write_json(job / "STATE.json", {"status": "inference", "arm": arm, "source": commit})
            audit[arm] = capture_arm(arm)
            gc.collect()
            import torch

            torch.cuda.empty_cache()
    write_json(job / "STATE.json", {"status": "aggregating", "source": commit})
    first, second = (audit[key] for key in arms)
    require(
        first["genes"] == second["genes"]
        and first["conditions"] == second["conditions"]
        and all(np.array_equal(first[k], second[k]) for k in ("truth", "controls", "direct")),
        "Two checkpoints do not share frozen gene, condition and control axes",
    )

In [ ]:
# ruff: noqa: RUF001
if mode == "audit":
    import csv

    import matplotlib.pyplot as plt
    from IPython.display import Markdown, display

    def save_csv(path, rows):
        with path.open("w", newline="") as stream:
            table = csv.DictWriter(stream, fieldnames=list(rows[0]))
            table.writeheader()
            table.writerows(rows)

    def share(part, whole):
        return float(part / whole) if whole > 0 else None

    def ranking(data):
        pred, true, control, direct = (data[k] for k in ("pred", "truth", "controls", "direct"))
        err = pred - true
        mag_error = np.square(np.abs(pred) - np.abs(true))
        over = np.square(np.maximum(np.abs(pred) - np.abs(true), 0))
        reversal = np.where(pred * true < 0, 4 * np.abs(pred * true), 0)
        require(
            np.allclose(mag_error + reversal, np.square(err), rtol=1e-12, atol=1e-12),
            "Elementwise error decomposition does not close",
        )
        total = over.sum()
        require(total > 0, "No over-amplification in frozen validation")

        def rms(x):
            return float(np.sqrt(np.mean(np.square(x))))

        def entries(axis, ids):
            weights = over.sum(axis=axis)
            result = []
            for i in np.argsort(-weights, kind="stable"):
                sl = (slice(None), i) if axis == 0 else (i, slice(None))
                expected, observed = true[sl], pred[sl]
                result.append(
                    {
                        "id": ids[i],
                        "overshoot_share": share(weights[i], total),
                        "overshoot_mse": float(over[sl].mean()),
                        "total_mse": float(np.square(err[sl]).mean()),
                        "truth_response_rms": rms(expected),
                        "pred_response_rms": rms(observed),
                        "mean_signed_residual": float(err[sl].mean()),
                        "mean_control_expression": float(control[sl].mean()),
                        "mean_truth_expression": float((expected + control[sl]).mean()),
                        "mean_pred_expression": float((observed + control[sl]).mean()),
                        "overshoot_pair_fraction": float((over[sl] > 0).mean()),
                        "direct_target_overshoot_share": share(
                            over[sl][direct[sl]].sum(), weights[i]
                        ),
                    }
                )
            return result

        genes = entries(0, data["genes"])
        conditions = entries(1, data["conditions"])
        pairs = []
        for k in np.argsort(-over.ravel(), kind="stable")[:100]:
            i, j = np.unravel_index(k, over.shape)
            pairs.append(
                {
                    "condition": data["conditions"][i],
                    "gene": data["genes"][j],
                    "overshoot_share": share(over[i, j], total),
                    "overshoot_squared": float(over[i, j]),
                    "prediction_delta": float(pred[i, j]),
                    "truth_delta": float(true[i, j]),
                    "control_mean": float(control[i, j]),
                    "direct_target": bool(direct[i, j]),
                }
            )

        truth_strength = np.abs(true)
        # Rank over the complete grid, keeping each quartile the same size.
        quartiles = np.array_split(np.argsort(truth_strength.ravel(), kind="stable"), 4)
        weak = [share(over.ravel()[ids].sum(), total) for ids in quartiles]
        condition_ratios = np.divide(
            np.linalg.norm(pred, axis=1),
            np.linalg.norm(true, axis=1),
            out=np.full(pred.shape[0], np.nan),
            where=np.linalg.norm(true, axis=1) > 0,
        )
        offset = np.square(err.mean(axis=0)).mean()
        varying = err.var(axis=0).mean()
        mse = np.square(err).mean()
        require(
            np.isclose(offset + varying, mse, rtol=1e-12, atol=1e-12),
            "Cross-condition offset decomposition does not close",
        )
        gene_control = control.mean(axis=0)
        gene_overshoot = over.mean(axis=0)
        return {
            "genes": genes,
            "conditions": conditions,
            "pairs": pairs,
            "offset": {
                "mean_gene_offset_mse": float(offset),
                "condition_varying_mse": float(varying),
                "mean_gene_offset_share": share(offset, mse),
                "over_amplification_share_of_mse": share(total, np.square(err).sum()),
                "magnitude_component_share": share(mag_error.sum(), np.square(err).sum()),
                "sign_reversal_component_share": share(reversal.sum(), np.square(err).sum()),
                "direct_target_overshoot_share": share(over[direct].sum(), total),
                "median_response_ratio": float(np.nanmedian(condition_ratios)),
                "top10_genes_overshoot_share": sum(row["overshoot_share"] for row in genes[:10]),
                "top10_conditions_overshoot_share": sum(
                    row["overshoot_share"] for row in conditions[:10]
                ),
                "truth_strength_quartile_overshoot_share": weak,
                "gene_control_spearman": float(
                    np.corrcoef(rank_average(gene_control), rank_average(gene_overshoot))[0, 1]
                ),
            },
        }

    def rank_average(values):
        order = np.argsort(values, kind="stable")
        _, starts, counts = np.unique(values[order], return_index=True, return_counts=True)
        ranks = np.empty(len(values), dtype=float)
        ranks[order] = np.repeat(starts + (counts - 1) / 2, counts)
        return ranks

    results = {arm: ranking(audit[arm]) for arm in arms}
    for arm, result in results.items():
        for dimension in ("genes", "conditions", "pairs"):
            save_csv(job / f"{arm}-{dimension}.csv", result[dimension])

    def top_panels(dimension, title, path):
        fig, panels = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
        for arm, axis in zip(arms, panels, strict=True):
            rows = results[arm][dimension][:12][::-1]
            names = [row.get("id", f"{row.get('gene')} | {row.get('condition')}") for row in rows]
            values = [100 * row["overshoot_share"] for row in rows]
            axis.barh(names, values, color="#227C84")
            axis.set(title=labels[arm], xlabel="Share of total over-amplification (%)")
            axis.grid(axis="x", alpha=0.24)
        fig.suptitle(title)
        fig.savefig(job / path, dpi=150, bbox_inches="tight")
        plt.show()
        plt.close(fig)

    top_panels("genes", "Unseen-expression genes driving over-amplification", "01-genes.png")
    top_panels("conditions", "Frozen perturbations driving over-amplification", "02-conditions.png")
    top_panels("pairs", "Largest gene by perturbation contributions", "03-pairs.png")

    figure, axes = plt.subplots(1, 3, figsize=(15, 4.3), constrained_layout=True)
    for offset, arm in enumerate(arms):
        r = results[arm]["offset"]
        x = np.arange(4) + 0.36 * (offset - 0.5)
        axes[0].bar(
            x,
            100 * np.array(r["truth_strength_quartile_overshoot_share"]),
            width=0.35,
            label=labels[arm],
        )
        axes[1].bar(
            np.arange(2) + 0.36 * (offset - 0.5),
            [100 * r["mean_gene_offset_share"], 100 * (1 - r["mean_gene_offset_share"])],
            width=0.35,
            label=labels[arm],
        )
        genes = results[arm]["genes"]
        axes[2].scatter(
            [g["mean_control_expression"] for g in genes],
            [g["overshoot_mse"] for g in genes],
            s=10,
            alpha=0.5,
            label=f"{labels[arm]} (rho {r['gene_control_spearman']:.2f})",
        )
    axes[0].set(
        title="True effect strength",
        ylabel="Over-amplification share (%)",
        xticks=range(4),
        xticklabels=["Q1 weak", "Q2", "Q3", "Q4 strong"],
    )
    axes[1].set(
        title="Residual decomposition",
        ylabel="MSE share (%)",
        xticks=range(2),
        xticklabels=["Gene offset", "Condition varying"],
    )
    axes[2].set(
        title="Baseline expression",
        xlabel="Mean paired control expression",
        ylabel="Mean squared over-amplification",
        yscale="symlog",
    )
    for axis in axes:
        axis.legend(fontsize=7)
        axis.grid(alpha=0.23)
    figure.savefig(job / "04-mechanism.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(figure)

    genes_overlap = set(r["id"] for r in results[arms[0]]["genes"][:10]) & {
        r["id"] for r in results[arms[1]]["genes"][:10]
    }
    conditions_overlap = set(r["id"] for r in results[arms[0]]["conditions"][:10]) & {
        r["id"] for r in results[arms[1]]["conditions"][:10]
    }
    text = [
        "# 全量 Jurkat 未见表达基因幅度审计",
        "",
        f"同一组冻结验证条件，共 {len(first['conditions'])} 个扰动和 "
        f"{len(first['genes'])} 个未见表达基因。每个扰动使用同一组 300 个对照。"
        "两组沿用末轮 checkpoint，Pearson 和 MSE 通过已验收回执的复跑容差核验。",
        "",
        "过度放大分数是预测绝对效应超过真实绝对效应的部分的平方。"
        "持续偏移分量是每个基因在全部验证扰动上的平均有符号残差的平方，反映跨条件共同偏移。",
        "",
    ]
    for arm in arms:
        r = results[arm]
        s = r["offset"]
        text += [
            f"## {labels[arm]}",
            "",
            f"预测与真实响应的幅度比中位数为 **{s['median_response_ratio']:.2f}×**。"
            f"共同偏移占总 MSE 的 **{s['mean_gene_offset_share']:.1%}**，其余来自随扰动变化的残差。"
            f"按真值效应强弱分成四等份后，最弱四分之一贡献了 "
            f"**{s['truth_strength_quartile_overshoot_share'][0]:.1%}** 的过度放大，"
            f"直接靶基因位置贡献 **{s['direct_target_overshoot_share']:.1%}**。",
            "",
            f"前 10 个基因合计贡献 **{s['top10_genes_overshoot_share']:.1%}**，"
            f"前 10 个扰动合计贡献 **{s['top10_conditions_overshoot_share']:.1%}**。",
            "",
            "| 排名 | 基因 | 过度放大占比 | 预测响应 RMS | 真实响应 RMS | 平均有符号误差 |",
            "| ---: | --- | ---: | ---: | ---: | ---: |",
        ]
        text += [
            f"| {i} | {g['id']} | {g['overshoot_share']:.2%} | "
            f"{g['pred_response_rms']:.4f} | {g['truth_response_rms']:.4f} | "
            f"{g['mean_signed_residual']:.4f} |"
            for i, g in enumerate(r["genes"][:10], 1)
        ]
        text += [
            "",
            "| 排名 | 扰动 | 过度放大占比 | 预测响应 RMS | 真实响应 RMS |",
            "| ---: | --- | ---: | ---: | ---: |",
        ]
        text += [
            f"| {i} | {q['id']} | {q['overshoot_share']:.2%} | {q['pred_response_rms']:.4f} | "
            f"{q['truth_response_rms']:.4f} |"
            for i, q in enumerate(r["conditions"][:10], 1)
        ]
        text += [""]
    text += [
        f"两组前 10 基因重合 {len(genes_overlap)} 个，前 10 扰动重合 {len(conditions_overlap)} 个。"
        "完整排名和贡献见 CSV。共同偏移、对照表达相关性和贡献排名均基于同一冻结验证集。"
    ]
    report = "\n".join(text) + "\n"
    (job / "REPORT_CN.md").write_text(report)
    display(Markdown(report))
    write_json(
        job / "SUMMARY.json",
        {
            "checkpoint_source": queue["evaluation_commit"],
            "notebook_source": commit,
            "notebook_sha256": sha256(notebook),
            "optimizer_updates": 0,
            "conditions": len(first["conditions"]),
            "unseen_expression_genes": len(first["genes"]),
            "arms": {arm: {**audit[arm]["provenance"], **results[arm]["offset"]} for arm in arms},
            "top10_common_genes": sorted(genes_overlap),
            "top10_common_conditions": sorted(conditions_overlap),
        },
    )
    outputs = [
        "SUMMARY.json",
        "REPORT_CN.md",
        "01-genes.png",
        "02-conditions.png",
        "03-pairs.png",
        "04-mechanism.png",
        *[f"{arm}-{dim}.csv" for arm in arms for dim in ("genes", "conditions", "pairs")],
    ]
    write_json(
        job / "STATE.json",
        {
            "status": "completed",
            "source": commit,
            "notebook_sha256": sha256(notebook),
            "files": {name: sha256(job / name) for name in outputs},
        },
    )

In [ ]:
if mode == "archive":
    import fcntl
    import re
    import shlex
    import subprocess
    import sys
    from concurrent.futures import ThreadPoolExecutor
    from contextlib import ExitStack
    from urllib.request import urlopen

    platform = Path(os.environ["GRADPERT_UNSEEN_PLATFORM"]).resolve()
    sys.path.insert(0, str(platform / "sdk"))
    from experiment_workspace import ExperimentPackage

    remote = os.environ["GRADPERT_UNSEEN_REMOTE"]
    require(
        re.fullmatch(
            r"/data/yilangliu/GraD-Pert/development/jurkat-unseen-gene-audit-[a-z0-9-]+", remote
        ),
        "Unexpected audit path",
    )
    user_home = Path.home()
    ssh = [
        "ssh",
        "-i",
        str(user_home / ".ssh/elanquant_yilangliu_ed25519"),
        "-S",
        "/tmp/elanquant-agent.sock",
        "-o",
        "IdentitiesOnly=yes",
        "-o",
        "BatchMode=yes",
        "-o",
        "ConnectTimeout=8",
        "-o",
        "ServerAliveInterval=30",
        "-o",
        "ServerAliveCountMax=3",
    ]
    host = "yilangliu@10.24.1.91"
    launch = read_json(job / "LAUNCH.json")
    remote_wait = """
import hashlib, json, os, select, sys
from pathlib import Path
directory, pid, ticks, command_sha = Path(sys.argv[1]), int(sys.argv[2]), sys.argv[3], sys.argv[4]
proc = Path(f"/proc/{pid}")
try:
    stat = (proc / "stat").read_text().rsplit(")", 1)[1].split()
    command = (proc / "cmdline").read_bytes()
    if stat[0] != "Z":
        if stat[19] != ticks or hashlib.sha256(command).hexdigest() != command_sha:
            raise ValueError("Audit process identity changed")
        with_fd = os.pidfd_open(pid)
        try:
            poller = select.poll()
            poller.register(with_fd, select.POLLIN)
            poller.poll()
        finally:
            os.close(with_fd)
except (FileNotFoundError, ProcessLookupError):
    pass
state = directory / "STATE.json"
if not state.is_file():
    raise RuntimeError("Audit exited without a terminal receipt")
result = json.loads(state.read_text())
if result["status"] != "completed":
    raise RuntimeError(f"Audit stopped in {result!r}")
print(json.dumps(result))
"""
    base = "http://127.0.0.1:8787"
    prefix = "unseen-gene-audit-20261111-"

    def get(path):
        with urlopen(base + path, timeout=30) as response:
            return response.read()

    def transfer(name, checksum):
        require(re.fullmatch(r"[a-zA-Z0-9._-]+", name), "Unexpected asset name")
        target = job / name
        partial = target.with_suffix(".incoming")
        args = ["rsync", "-ti", "-e", shlex.join(ssh), f"{host}:{remote}/{name}", str(partial)]
        if not target.exists():
            subprocess.run([args[0], "--dry-run", *args[1:]], check=True, stdout=subprocess.DEVNULL)
            subprocess.run(
                [args[0], "--max-size=2097152", *args[1:]], check=True, stdout=subprocess.DEVNULL
            )
            require(
                partial.is_file() and sha256(partial) == checksum, "Received file hash mismatch"
            )
            partial.replace(target)
        require(
            sha256(target) == checksum and target.stat().st_size <= 2 * 1024**2,
            "Cached asset hash mismatch",
        )
        return target

    def preserve(asset, package):
        if asset.get("file") and asset["file"] not in package.assets:
            content = get("/api/asset/" + asset["file"])
            require(
                hashlib.sha256(content).hexdigest() == asset["sha256"],
                "Existing asset checksum changed",
            )
            package.assets[asset["file"]] = content

    def verify(asset):
        require(
            hashlib.sha256(get("/api/asset/" + asset["file"])).hexdigest() == asset["sha256"],
            "Imported asset checksum differs",
        )

    try:
        args = [remote, str(launch["pid"]), launch["start_ticks"], launch["command_sha256"]]
        command = shlex.join(["python3", "-", *args])
        terminal = json.loads(
            subprocess.check_output([*ssh, host, command], input=remote_wait.encode())
        )
        require(
            terminal["source"] == launch["source"]
            and terminal["notebook_sha256"] == launch["notebook_sha256"],
            "Audit receipt differs",
        )
        write_json(job / "SERVER_TERMINAL.json", terminal)
        shared = (
            "SUMMARY.json",
            "REPORT_CN.md",
            "01-genes.png",
            "02-conditions.png",
            "03-pairs.png",
            "04-mechanism.png",
        )
        expected = set(shared) | {
            f"{arm}-{d}.csv" for arm in arms for d in ("genes", "conditions", "pairs")
        }
        require(set(terminal["files"]) == expected, "Unexpected server output set")
        paths = {name: transfer(name, digest) for name, digest in terminal["files"].items()}
        report = read_json(paths["SUMMARY.json"])
        require(
            report["conditions"] == 445
            and report["optimizer_updates"] == 0
            and report["notebook_sha256"] == launch["notebook_sha256"],
            "Audit summary differs",
        )
        package = ExperimentPackage("grad-pert", "GraD-Pert", "placeholder")
        package.package = {
            "schema_version": "1.0",
            "project": {"id": "grad-pert", "name": "GraD-Pert"},
            "experiments": [],
        }
        assets_by_arm = {}
        before = {}
        lock_dir = platform / ".local/platform-workflow-locks"
        lock_dir.mkdir(parents=True, exist_ok=True)
        with ExitStack() as lock_stack:
            for eid in sorted(report["arms"][arm]["experiment_id"] for arm in arms):
                handle = lock_stack.enter_context((lock_dir / f"{eid}.lock").open("a"))
                fcntl.flock(handle, fcntl.LOCK_EX)
            for arm in arms:
                info = report["arms"][arm]
                route = "/api/export?project=grad-pert&id=" + info["experiment_id"]
                payload = json.loads(get(route))
                (experiment,) = payload["experiments"]
                (run,) = experiment["runs"]
                require(
                    run["id"] == info["run_id"]
                    and run["checkpoint_sha256"] == info["checkpoint_sha256"],
                    "Platform checkpoint differs",
                )
                standard = next(
                    d for d in run["analyses"] if d["id"] == "frozen-expression-and-training"
                )
                standard["artifacts"] = [
                    x for x in standard["artifacts"] if not x["id"].startswith(prefix)
                ]
                before[arm] = json.loads(json.dumps(experiment))
                for asset in [
                    *run["artifacts"],
                    *(a for d in run["analyses"] for a in d["artifacts"]),
                ]:
                    preserve(asset, package)
                files = [*shared, *(f"{arm}-{d}.csv" for d in ("genes", "conditions", "pairs"))]
                assets = [
                    package.artifact(
                        paths[name],
                        artifact_id=prefix + Path(name).stem,
                        name="未见表达放大诊断 · " + name,
                        kind="image"
                        if name.endswith(".png")
                        else ("table" if name.endswith(".csv") else "report"),
                    )
                    for name in files
                ]
                standard["artifacts"].extend(assets)
                assets_by_arm[arm] = (route, {a["id"]: a for a in assets})
                package.package["project"]["name"] = payload["project"]["name"]
                package.package["experiments"].append(experiment)
            package_file = package.write(job / "package")
            import_cmd = [
                "node",
                str(platform / "scripts/import-package.mjs"),
                str(package_file),
                f"--base={base}",
            ]
            subprocess.run([*import_cmd, "--check-only"], check=True)
            subprocess.run([*import_cmd, "--replace"], check=True)
            for arm, (route, expected_assets) in assets_by_arm.items():
                (live,) = json.loads(get(route))["experiments"]
                (run,) = live["runs"]
                standard = next(
                    d for d in run["analyses"] if d["id"] == "frozen-expression-and-training"
                )
                actual = {a["id"]: a for a in standard["artifacts"] if a["id"].startswith(prefix)}
                require(actual == expected_assets, "Imported attachments differ from package")
                with ThreadPoolExecutor(max_workers=4) as pool:
                    list(pool.map(verify, actual.values()))
                standard["artifacts"] = [a for a in standard["artifacts"] if a["id"] not in actual]
                require(live == before[arm], "Platform changed the original experiment record")
        write_json(
            job / "READY.json",
            {
                "status": "completed",
                "report": str(paths["REPORT_CN.md"]),
                "summary_sha256": sha256(paths["SUMMARY.json"]),
                "verified_platform_attachments": {
                    arm: len(x[1]) for arm, x in assets_by_arm.items()
                },
            },
        )
    except Exception as error:
        write_json(job / "FAILED.json", {"status": "failed", "error": str(error)})
        raise